For this assignment you will use a gridsearch algorithm, such as the particle swarm or CSO to tune hyperparameters for a Pytorch neural network design, such as Alex Net, to create a data application for the CiFAR10  data set and yield good accuracy on the test set. For CiFAR10, good accuracy on the test set is over 84%.

Alternatively, if you prefer to work with EMNIST (https://www.nist.gov/itl/products-and-services/emnist-dataset) you should aim at an accuracy over 90%.

Reference for EMNIST: https://arxiv.org/pdf/1702.05373.pdf

***For my assignment, I will be using the CiFAR10 dataset as we did in class, specifically implementing a gridsearch algorithm for altering hyperparameters. First I will try using particle swarm.  I will be taking from the Module 5 and 6 notebooks, and I will specify where I am getting the information from as well as how I'm adjusting my code to work on a good model.***

In [129]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torch.utils.data.sampler import SubsetRandomSampler

In [130]:
# Get gpu, mps or cpu device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")


Using cuda device


In [131]:
#Looking at Module 5 notebook, note that the hyperparameters that we want to test and iterate through are these:
num_classes = 10
num_epochs = 5
batch_size = 35
learning_rate = 0.005

#This function is going to create the training and validation data loaders with data augumentation if specified.
def get_train_valid_loader(data_dir,
                           batch_size,
                           augment,
                           random_seed,
                           valid_size=0.1,
                           shuffle=True):
    normalize = transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010],
    )

    # define transforms
    valid_transform = transforms.Compose([
            transforms.Resize((128,128)),
            transforms.ToTensor(),
            normalize,
    ])
    if augment:
        train_transform = transforms.Compose([
            transforms.Resize((128,128)),
            transforms.RandomCrop(120, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize,
        ])
    else:
        train_transform = transforms.Compose([
            transforms.Resize((227,227)),
            transforms.ToTensor(),
            normalize,
        ])

    # load the dataset
    train_dataset = datasets.CIFAR10(
        root=data_dir, train=True,
        download=True, transform=train_transform,
    )

    valid_dataset = datasets.CIFAR10(
        root=data_dir, train=True,
        download=True, transform=valid_transform,
    )

    num_train = len(train_dataset)
    indices = list(range(num_train))
    split = int(np.floor(valid_size * num_train))

    if shuffle:
        np.random.seed(random_seed)
        np.random.shuffle(indices)

    train_idx, valid_idx = indices[split:], indices[:split]
    train_sampler = SubsetRandomSampler(train_idx)
    valid_sampler = SubsetRandomSampler(valid_idx)

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, sampler=train_sampler)

    valid_loader = torch.utils.data.DataLoader(
        valid_dataset, batch_size=batch_size, sampler=valid_sampler)

    return (train_loader, valid_loader)

#This function is going to create the data loader for the test dataset.
def get_test_loader(data_dir,
                    batch_size,
                    shuffle=True):
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )

    # define transform
    transform = transforms.Compose([
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        normalize,
    ])

    dataset = datasets.CIFAR10(
        root=data_dir, train=False,
        download=True, transform=transform,
    )

    data_loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle
    )
    return data_loader

train_loader, valid_loader = get_train_valid_loader(data_dir = './data',                                      batch_size = 64,
                       augment = True,random_seed = 123)

test_loader = get_test_loader(data_dir = './data',
                              batch_size = 72)


#Now I will be taking from the Pytorch neural network design AlexNet,that we went over in an example in Module 5
#This is going to accept input images, typically RGB images, with a fixed size.
#Demonstrates the power of deep convolutional neural networks for image classification tasks!

class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 3, stride = 2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 3, stride = 2))
        self.layer3 = nn.Sequential(
            nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(384),
            nn.ReLU())
        self.layer4 = nn.Sequential(
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(384),
            nn.ReLU())
        self.layer5 = nn.Sequential(
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 3, stride = 2))
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1024, 4096),
            nn.ReLU())
        self.fc1 = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU())
        self.fc2= nn.Sequential(
            nn.Linear(4096, num_classes))

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.layer5(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        out = self.fc1(out)
        out = self.fc2(out)
        return out



Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


In [132]:
#Therefore, I will create a function that does through a specified amount of epochs
#Then ultimately, this will train the AlexNet model to hopefully give us a good accuracy for predicting the pictures in the CIFAR10 Dataset!

def train_model(num_classes, num_epochs, batch_size, lr, train_loader, valid_loader):
  # updated_accuracy = 0
  # updated_learning_rate = learning_rate
  model = AlexNet().to(device)
  # Loss and optimizer
  criterion = nn.CrossEntropyLoss()
  model = AlexNet().to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr = lr)# train model
  # Train the model
  total_step = len(train_loader)

  for epoch in range(num_epochs):
      #accuracy = 0
      #learning_rate = 0
      for i, (images, labels) in enumerate(train_loader):
          # Move tensors to the configured device
          images = images.to(device)
          labels = labels.to(device)

          # Forward pass
          outputs = model(images)
          loss = criterion(outputs, labels)

          # Backward and optimize
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

      #print(epoch)
      print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
                    .format(epoch+1, num_epochs, i+1, total_step, loss.item()))

  # Validation
  with torch.no_grad():
      correct = 0
      total = 0
      for images, labels in valid_loader:
          images = images.to(device)
          labels = labels.to(device)
          outputs = model(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()
          del images, labels, outputs

      print('Accuracy of the network on the {} validation images: {} %'.format(5000, 100 * correct / total))
      accuracy = 100 * correct / total
      return accuracy


In [133]:
#Prepping for the grid search with the obj function
def objective_function(hyperparameters, validation):
    num_classes = 10
    num_epochs = 5
    learning_rate = 0.005
    lr, batch_size = hyperparameters
    return -train_model(10, num_epochs, batch_size, lr, train_loader, validation)

#From the Grid Search notebook!
def particle_swarm_optimization(num_dimensions, num_particles, max_iter, validation, i_min = -10, i_max = 10, w = 0.5, c1 = 0.25, c2 = 0.75, bounds=None):
    # Initialize the particles
    # This creates a data structure such as a dictionary
    particles = [({'position': [np.random.uniform(bounds[i][0], bounds[i][1]) for i in range(num_dimensions)],
                'velocity': [np.random.uniform(-1, 1) for _ in range(num_dimensions)],
                'pbest': float('inf'),
                'pbest_position': None})
                for _ in range(num_particles)]

    # Initialize global best
    gbest_value = float('inf')
    gbest_position = None

    for _ in range(max_iter):
        for particle in particles:
            position = particle['position']
            velocity = particle['velocity']

            # Calculate the current value
            current_value = objective_function(position, validation)

            # Update personal best
            if current_value < particle['pbest']:
                particle['pbest'] = current_value
                particle['pbest_position'] = position.copy()

            # Update global best
            if current_value < gbest_value:
                gbest_value = current_value
                gbest_position = position.copy()

            # Update particle's velocity and position
            for i in range(num_dimensions):
                r1, r2 = np.random.uniform(), np.random.uniform()
                velocity[i] = w * velocity[i] + c1*r1 * (particle['pbest_position'][i] - position[i]) + c2*r2 * (gbest_position[i] - position[i])
                position[i] += velocity[i]
                # legalize the values to the provided bounds
                if bounds is not None:
                    position[i] = np.clip(position[i],bounds[i][0],bounds[i][1])

    return gbest_position, gbest_value



In [134]:
bounds = [(0.0001, 0.001), (32, 64)] #lr, batch_size
best_hyperparameters, best_accuracy = particle_swarm_optimization(num_dimensions=2,
                                                                   num_particles=3,
                                                                   max_iter=50,
                                                                   validation = test_loader,
                                                                   bounds=bounds
                                                                   )

print(f"{best_hyperparameters[0]}, {int(best_hyperparameters[1])}")
print(f"{best_accuracy}")


Epoch [1/5], Step [704/704], Loss: 1.3583
Epoch [2/5], Step [704/704], Loss: 1.2346
Epoch [3/5], Step [704/704], Loss: 1.0687
Epoch [4/5], Step [704/704], Loss: 0.9857
Epoch [5/5], Step [704/704], Loss: 1.9672
Accuracy of the network on the 5000 validation images: 73.32 %
Epoch [1/5], Step [704/704], Loss: 1.6377
Epoch [2/5], Step [704/704], Loss: 2.0050
Epoch [3/5], Step [704/704], Loss: 0.8206
Epoch [4/5], Step [704/704], Loss: 0.3549
Epoch [5/5], Step [704/704], Loss: 1.4418
Accuracy of the network on the 5000 validation images: 72.82 %
Epoch [1/5], Step [704/704], Loss: 0.8330
Epoch [2/5], Step [704/704], Loss: 0.9262
Epoch [3/5], Step [704/704], Loss: 0.5433
Epoch [4/5], Step [704/704], Loss: 1.1405
Epoch [5/5], Step [704/704], Loss: 0.7239
Accuracy of the network on the 5000 validation images: 73.14 %
Epoch [1/5], Step [704/704], Loss: 1.2169
Epoch [2/5], Step [704/704], Loss: 1.7936
Epoch [3/5], Step [704/704], Loss: 0.8781
Epoch [4/5], Step [704/704], Loss: 0.3637
Epoch [5/5], 

KeyboardInterrupt: 

In [ ]:
#I had to stop the loop but it was infinite and kept going

##Concluding Remarks

There is a noticeable trend of decreasing loss values over the epochs, indicating that my neural network is effectively learning from the data. However, I'm somewhat dissatisfied with the accuracy, which hovered around 76%.

Achieving this level of accuracy required numerous trials, with the initial results barely exceeding 50%. I leveraged functions from Module 5 and Module 6, tweaking parameters to understand their impact on the model. Upon discovering that the first epoch achieved an accuracy of about 70%, I was ecstatic and let the model continue running. However, it appeared that the num_epochs parameter did not register properly, causing the model to run repeatedly for about 40-50 minutes without progress. Unfortunately, I had to cancel the cell and analyze the epochs I had completed. As a result, the cell never completed, preventing me from seeing the exact best accuracy. Thankfully, I have an approximate estimate.

In [ ]:
##Concluding Remarks